# Waste Segmentation Evaluation Notebook (Colab-ready)This notebook prepares an evaluation pipeline to compare segmentation models on the TACO dataset (or your custom dataset).It converts TACO -> COCO, runs SAM automatic mask generator, evaluates with pycocotools, demonstrates a Mask R-CNN baseline,and creates a comparison table + plots. Run on Google Colab.

## 1) Setup (Run on Colab)Install dependencies and upload dataset/checkpoints. Follow instructions in the cell.

In [ ]:
# Colab setup - run this cell in Google Colabimport sysprint('Python', sys.version)# Install common dependencies!pip install -q pycocotools==2.0.6!pip install -q matplotlib tqdm imagehash scikit-image opencv-python-headless!pip install -q git+https://github.com/facebookresearch/segment-anything.gitprint('NOTE: Detectron2 installation may require matching CUDA runtime; install detectron2 per your Colab CUDA version.')print('Upload your dataset (taco.zip or taco/ folder) and SAM checkpoint (sam_vit_l.pth) via Colab file UI or mount Drive.')

## 2) Helper utilitiesUtility functions for conversion, JSON saving, and simple visualization.

In [ ]:
import os, jsonfrom pathlib import Pathfrom PIL import Imageimport numpy as npimport cv2from tqdm import tqdmdef save_json(obj, path):    with open(path, 'w') as f:        json.dump(obj, f)def load_json(path):    with open(path, 'r') as f:        return json.load(f)print('helpers loaded')

## 3) Convert TACO to COCO formatAdapt this converter if your TACO JSON layout differs. Place taco JSON and images in the working directory.

In [ ]:
def taco_to_coco(taco_ann_path, images_dir, out_coco_path):    taco = load_json(taco_ann_path)    images = {}    annotations = []    categories = {}    img_id_map = {}    next_img_id = 1    next_ann_id = 1    for ann in taco:        img_name = ann.get('image_id') or ann.get('image_filename') or ann.get('filename')        if img_name is None:            continue        if img_name not in img_id_map:            img_id_map[img_name] = next_img_id            img_path = os.path.join(images_dir, img_name)            if not os.path.exists(img_path):                print('Warning: image not found', img_path)            im = Image.open(img_path)            w,h = im.size            images[next_img_id] = {'id': next_img_id, 'file_name': img_name, 'width': w, 'height': h}            next_img_id += 1        img_id = img_id_map[img_name]        cls = ann.get('class') or ann.get('category') or ann.get('label')        if cls not in categories:            categories[cls] = len(categories)+1        cat_id = categories[cls]        poly = ann.get('poly') or ann.get('segmentation')        if poly is None:            bbox = ann.get('bbox')            if bbox is None:                continue            x,y,w,h = bbox            seg = [x,y,x+w,y,x+w,y+h,x,y+h]        else:            seg = poly            if isinstance(seg, list) and len(seg)>0 and isinstance(seg[0], list):                seg = sum(seg, [])        xs = seg[0::2]        ys = seg[1::2]        x_min = float(min(xs)); x_max = float(max(xs))        y_min = float(min(ys)); y_max = float(max(ys))        bbox = [x_min, y_min, x_max-x_min, y_max-y_min]        annotations.append({'id': next_ann_id, 'image_id': img_id, 'category_id': cat_id, 'segmentation': [seg], 'bbox': bbox, 'area': bbox[2]*bbox[3], 'iscrowd': 0})        next_ann_id += 1    coco = {'images': list(images.values()), 'annotations': annotations, 'categories': [{'id': cid, 'name': name} for name,cid in categories.items()]}    save_json(coco, out_coco_path)    print('Saved COCO to', out_coco_path)print('taco->coco converter ready')

## 4) Run SAM automatic mask generator and produce COCO-style resultsPlace your SAM checkpoint (e.g., sam_vit_l.pth) in the working directory. This cell will run SAM's AutomaticMaskGenerator and write RLE-encoded masks to a results JSON.

In [ ]:
import torchfrom pycocotools import mask as mask_utilsfrom segment_anything import sam_model_registry, SamAutomaticMaskGeneratorimport numpy as npdef binary_mask_to_rle(binary_mask):    rle = mask_utils.encode(np.asfortranarray(binary_mask.astype('uint8')))    rle['counts'] = rle['counts'].decode('ascii')    return rledef run_sam_auto_and_write_coco(images_dir, coco_images, sam_checkpoint, output_results_json='sam_results.json', min_mask_area=100):    device = 'cuda' if torch.cuda.is_available() else 'cpu'    sam = sam_model_registry['vit_l'](checkpoint=sam_checkpoint).to(device)    mask_generator = SamAutomaticMaskGenerator(sam)    results = []    for img_info in tqdm(coco_images):        img_path = os.path.join(images_dir, img_info['file_name'])        image = cv2.imread(img_path)        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)        masks = mask_generator.generate(image)        for m in masks:            seg = m['segmentation']            area = int(seg.sum())            if area < min_mask_area:                continue            rle = binary_mask_to_rle(seg)            results.append({'image_id': img_info['id'], 'category_id': 1, 'segmentation': rle, 'score': float(m.get('predicted_iou', 0.0))})    save_json(results, output_results_json)    print('Wrote', output_results_json, 'with', len(results), 'masks')print('SAM auto mask -> COCO results cell ready. Configure paths and run in Colab.')

## 5) Evaluate COCO mask predictions with pycocotoolsThis evaluates the `results.json` predictions against ground truth `instances_test.json`.

In [ ]:
from pycocotools.coco import COCOfrom pycocotools.cocoeval import COCOevaldef coco_eval(gt_json, results_json, iou_type='segm'):    cocoGt = COCO(gt_json)    cocoDt = cocoGt.loadRes(results_json)    cocoEval = COCOeval(cocoGt, cocoDt, iouType=iou_type)    cocoEval.evaluate()    cocoEval.accumulate()    cocoEval.summarize()    return cocoEval.statsprint('COCO eval function ready. Example: coco_eval("instances_test.json","sam_results.json")')

## 6) Example: Mask R-CNN baseline (Detectron2)Detectron2 may need manual installation depending on CUDA. If installed, use the registration helper and predictor shown below.

In [ ]:
try:    import detectron2    from detectron2.utils.logger import setup_logger    setup_logger()    from detectron2.data.datasets import register_coco_instances    from detectron2.engine import DefaultPredictor    from detectron2.config import get_cfg    from detectron2 import model_zoo    print('Detectron2 imported')except Exception as e:    print('Detectron2 not available in this runtime. Install per your CUDA/version. Error:', e)def setup_detectron2_for_eval(coco_train_json, coco_val_json, images_dir, dataset_name='taco'):    register_coco_instances(dataset_name + '_train', {}, coco_train_json, images_dir)    register_coco_instances(dataset_name + '_val', {}, coco_val_json, images_dir)    print('Registered Detectron2 datasets')def detectron_infer_example(weights_path, cfg_name='COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml', score_thresh=0.5):    cfg = get_cfg()    cfg.merge_from_file(model_zoo.get_config_file(cfg_name))    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = score_thresh    cfg.MODEL.WEIGHTS = weights_path    predictor = DefaultPredictor(cfg)    return predictorprint('Detectron2 setup cell ready')

## 7) Postprocessing & Table generationLoad evaluation numbers into `eval_results` dict and generate comparison table and plots.

In [ ]:
import pandas as pdimport matplotlib.pyplot as pltdef make_comparison_table(eval_results, out_csv='segmentation_comparison.csv'):    df = pd.DataFrame.from_dict(eval_results, orient='index')    df.to_csv(out_csv)    display(df)    return dfdef plot_metrics(df, metrics=['mIoU','AP_mask','Inference_ms']):    for m in metrics:        if m in df.columns:            plt.figure(figsize=(8,4))            df[m].sort_values(ascending=False).plot(kind='bar')            plt.title(m)            plt.ylabel(m)            plt.tight_layout()            plt.show()print('Table & plotting utilities ready')

## 8) Usage instructions (Colab)1. Upload `taco/` dataset or `instances_test.json` and images.2. Upload `sam_vit_l.pth` checkpoint to working directory.3. Run SAM cell to generate `sam_results.json`.4. Run COCO eval cell: `coco_eval('instances_test.json','sam_results.json')`.5. For Detectron2 baseline, install detectron2 and run training/eval or load a pretrained Mask R-CNN and evaluate.

## 9) Demo: populate eval_results (placeholder)After running your experiments, populate `eval_results` with your measured metrics.

In [ ]:
eval_results = {    'SAM_vit_l_zero_shot': {'mIoU': 0.45, 'AP_mask': 0.21, 'Inference_ms': 350.0, 'Model_MB': 1200},    'SAM_vit_l_finetuned': {'mIoU': 0.62, 'AP_mask': 0.39, 'Inference_ms': 370.0, 'Model_MB': 1200},    'MaskRCNN_R50': {'mIoU': 0.55, 'AP_mask': 0.33, 'Inference_ms': 120.0, 'Model_MB': 250},    'DeepLabv3plus': {'mIoU': 0.48, 'AP_mask': 0.25, 'Inference_ms': 140.0, 'Model_MB': 300},    'FastSAM': {'mIoU': 0.50, 'AP_mask': 0.30, 'Inference_ms': 90.0, 'Model_MB': 80}}df = make_comparison_table(eval_results)plot_metrics(df)

## 10) Downloading the NotebookAfter running in Colab you can download via `File > Download .ipynb` or copy the generated file from the runtime to your Drive.